# Beyond Idealized Settings — experiment notebook

**Run order (top to bottom):**

1. **Shared library** — run once. Defines everything below.
2. **Step 1 (baseline)** — run **three times**: `WHICH="smoke"`, then `"noniid"`, then `"iid"`. (Both `noniid` and `iid` are needed for the IID-vs-non-IID axis.)
3. **Step 2** — training-stage axis (DLG & IG) + attack-correctness check.
4. **Step 3** — batch-size axis (DLG & IG).
5. **Step 4** — IID vs non-IID axis.
6. **Step 5** — single-step vs multi-step axis.
7. **Step 6** — defense × attack matrix (run before Step 7).
8. **Step 7** — defense utility + privacy–utility trade-off.

Kaggle: Accelerator = **GPU T4**, Internet = **On**. Outputs (figures, `.pt` files) land in `/kaggle/working/outputs`. Sessions wipe on close — click **Save Version** or download what you need.

All numbers are for a **single victim image** (fast pilot). The multi-image averaging pass comes later.


In [1]:
# ==========================================================================
# SHARED LIBRARY  --  RUN THIS CELL FIRST (every other cell depends on it)
# Beyond Idealized Settings: GIA & Defense Evaluation in Non-IID Cross-Silo FL
# Defines: config, data, model, metrics, gradient capture, defenses, attacks,
# FedAvg (clean + defended w/ error feedback). Nothing is redefined below.
# ==========================================================================
import os, time, copy, random
import numpy as np
import torch, torch.nn as nn
import torchvision, torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
from dataclasses import dataclass
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as _psnr
from skimage.metrics import structural_similarity as _ssim
from scipy.optimize import linear_sum_assignment

OUT = "/kaggle/working/outputs"
RESULTS = {}                      # later cells read earlier cells' results from here

def resolve_device(): return "cuda" if torch.cuda.is_available() else "cpu"
DEVICE = resolve_device()

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

@dataclass
class Config:
    num_clients:int=10; alpha:float=0.5; rounds:int=60; local_epochs:int=2
    local_bs:int=64; lr:float=0.1; num_classes:int=10
    data_root:str="/kaggle/working/data"; data_fraction:float=1.0; seed:int=42

# ----------------------------- data -----------------------------
def get_datasets(cfg):
    tf=T.Compose([T.ToTensor()])
    tr=torchvision.datasets.CIFAR10(cfg.data_root,train=True,download=True,transform=tf)
    te=torchvision.datasets.CIFAR10(cfg.data_root,train=False,download=True,transform=tf)
    return tr,te

def dirichlet_partition(labels,n,alpha,k,seed=42):
    rng=np.random.default_rng(seed); labels=np.array(labels); idx=[[] for _ in range(n)]
    for c in range(k):
        ic=np.where(labels==c)[0]; rng.shuffle(ic)
        pr=rng.dirichlet(alpha*np.ones(n)); cut=(np.cumsum(pr)*len(ic)).astype(int)[:-1]
        for i,part in enumerate(np.split(ic,cut)): idx[i].extend(part.tolist())
    for i in range(n): rng.shuffle(idx[i])
    return idx

def make_client_loaders(cfg,train):
    labels=train.targets
    if cfg.data_fraction<1.0:
        rng=np.random.default_rng(cfg.seed)
        keep=rng.choice(len(labels),int(len(labels)*cfg.data_fraction),replace=False)
        sub=[labels[i] for i in keep]
        parts=dirichlet_partition(sub,cfg.num_clients,cfg.alpha,cfg.num_classes,cfg.seed)
        parts=[[keep[j] for j in p] for p in parts]
    else:
        parts=dirichlet_partition(labels,cfg.num_clients,cfg.alpha,cfg.num_classes,cfg.seed)
    loaders=[DataLoader(Subset(train,p),batch_size=cfg.local_bs,shuffle=True) for p in parts]
    return loaders,parts

# ----------------------------- model -----------------------------
class LeNet(nn.Module):
    def __init__(self,num_classes=10):
        super().__init__(); act=nn.Sigmoid
        self.body=nn.Sequential(
            nn.Conv2d(3,12,5,padding=2,stride=2),act(),
            nn.Conv2d(12,12,5,padding=2,stride=2),act(),
            nn.Conv2d(12,12,5,padding=2,stride=1),act())
        self.fc=nn.Linear(12*8*8,num_classes)
    def forward(self,x):
        x=self.body(x); return self.fc(x.view(x.size(0),-1))

def build_model(): return LeNet(10).to(DEVICE)

def weights_init(m):     # DLG/iDLG init; ONLY for the idealized-validation sanity check
    if hasattr(m,"weight") and m.weight is not None: m.weight.data.uniform_(-0.5,0.5)
    if hasattr(m,"bias") and m.bias is not None: m.bias.data.uniform_(-0.5,0.5)

# ----------------------------- metrics -----------------------------
def _pair(recon,true):          # recon,true: (3,32,32) tensors in [0,1]
    r=recon.detach().clamp(0,1).permute(1,2,0).cpu().numpy()
    t=true.detach().clamp(0,1).permute(1,2,0).cpu().numpy()
    p=_psnr(t,r,data_range=1.0)
    try: s=_ssim(t,r,data_range=1.0,channel_axis=2)
    except TypeError: s=_ssim(t,r,data_range=1.0,multichannel=True)
    return float(p),float(s),float(((r-t)**2).mean())

def score(recon_batch,true_batch):    # (B,3,32,32) each; Hungarian match; mean PSNR,SSIM,MSE
    B=recon_batch.shape[0]; cost=np.zeros((B,B))
    for i in range(B):
        for j in range(B):
            cost[i,j]=((recon_batch[i]-true_batch[j])**2).mean().item()
    r,c=linear_sum_assignment(cost); ps,ss,ms=[],[],[]
    for i,j in zip(r,c):
        p,s,m=_pair(recon_batch[i],true_batch[j]); ps.append(p); ss.append(s); ms.append(m)
    return float(np.mean(ps)),float(np.mean(ss)),float(np.mean(ms))

def total_variation(x):
    dh=(x[:,:,1:,:]-x[:,:,:-1,:]).abs().mean(); dw=(x[:,:,:,1:]-x[:,:,:,:-1]).abs().mean()
    return dh+dw

def extract_label(real_grad):         # iDLG analytic label (single image)
    return int(real_grad[-1].argmin().item())

# ----------------------------- gradient capture -----------------------------
def capture_gradient(state,X,Y):      # single-step gradient of batch (X,Y)
    m=build_model(); m.load_state_dict(state); m.eval(); m.zero_grad()
    loss=nn.CrossEntropyLoss()(m(X.to(DEVICE)),Y.to(DEVICE))
    g=torch.autograd.grad(loss,m.parameters())
    return [t.detach().to(DEVICE).clone() for t in g]

def capture_multistep_update(state,X,Y,local_steps,lr):
    # victim does `local_steps` SGD steps on (X,Y) from `state`; returns pseudo-grad = -delta/lr.
    # local_steps=1 reproduces the single-step gradient EXACTLY; larger = harder (FedAvg).
    m=build_model(); m.load_state_dict(state); m.train(); params=list(m.parameters())
    theta0=[p.detach().clone() for p in params]; crit=nn.CrossEntropyLoss()
    for _ in range(local_steps):
        loss=crit(m(X.to(DEVICE)),Y.to(DEVICE)); g=torch.autograd.grad(loss,params)
        with torch.no_grad():
            for p,gg in zip(params,g): p-=lr*gg
    delta=[(t0-p.detach()) for t0,p in zip(theta0,params)]      # = lr * sum_t grad_t
    return [d/lr for d in delta]

# ----------------------------- defenses -----------------------------
def prune_gradient(grad,ratio):
    if ratio<=0: return [g.clone() for g in grad]
    flat=torch.cat([g.flatten().abs() for g in grad])
    k=max(1,min(int(len(flat)*ratio),len(flat)-1)); thr=torch.kthvalue(flat,k).values
    return [g*(g.abs()>thr).float() for g in grad]

def dp_gradient(grad,clip_C,sigma):
    if clip_C is not None:
        norm=torch.sqrt(sum((g**2).sum() for g in grad)); scale=min(1.0,float(clip_C)/float(norm+1e-12))
    else: scale=1.0
    return [g*scale+(torch.randn_like(g)*sigma if sigma else 0.0) for g in grad]

def apply_defense(grad,spec):
    g=grad
    if spec.get("prune",0)>0: g=prune_gradient(g,spec["prune"])
    if spec.get("clip") is not None or spec.get("sigma",0)>0: g=dp_gradient(g,spec.get("clip"),spec.get("sigma",0))
    return g

def grad_norm(grad): return float(torch.sqrt(sum((g**2).sum() for g in grad)).item())

# ----------------------------- attacks (B images; B=1 = single image) -----------------------------
def dlg_attack(state,real_grad,labels,iters=300,lr=1.0,restarts=2,early_stop=1e-9):
    B=labels.shape[0]; m=build_model(); m.load_state_dict(state); m.eval()
    for p in m.parameters(): p.requires_grad_(True)
    real_grad=[g.detach().to(DEVICE) for g in real_grad]; lab=labels.to(DEVICE); crit=nn.CrossEntropyLoss()
    bL,bI=float("inf"),None
    for _ in range(restarts):
        latent=torch.randn(B,3,32,32,device=DEVICE,requires_grad=True)
        def gm():
            dx=torch.sigmoid(latent); loss=crit(m(dx),lab)
            dg=torch.autograd.grad(loss,m.parameters(),create_graph=True)
            return sum(((a-b)**2).sum() for a,b in zip(dg,real_grad))
        opt=torch.optim.LBFGS([latent],lr=lr,line_search_fn="strong_wolfe")
        l,i=float("inf"),torch.sigmoid(latent).detach().clone()
        for it in range(iters):
            def closure(): opt.zero_grad(); d=gm(); d.backward(); return d
            d=opt.step(closure); v=float(d.item())
            if np.isfinite(v) and v<l: l,i=v,torch.sigmoid(latent).detach().clone()
            if (not np.isfinite(v)) or v<early_stop: break
        if l<bL: bL,bI=l,i
    return bI

def ig_attack(state,real_grad,labels,iters=1500,lr=0.1,tv_weight=1e-2,restarts=2):
    B=labels.shape[0]; m=build_model(); m.load_state_dict(state); m.eval()
    for p in m.parameters(): p.requires_grad_(True)
    real_grad=[g.detach().to(DEVICE) for g in real_grad]; rn=torch.sqrt(sum((g**2).sum() for g in real_grad))
    lab=labels.to(DEVICE); crit=nn.CrossEntropyLoss(); bL,bI=float("inf"),None
    for _ in range(restarts):
        latent=torch.randn(B,3,32,32,device=DEVICE,requires_grad=True)
        opt=torch.optim.Adam([latent],lr=lr); sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=iters)
        l,i=float("inf"),torch.sigmoid(latent).detach().clone()
        for it in range(iters):
            opt.zero_grad(); dx=torch.sigmoid(latent); loss=crit(m(dx),lab)
            dg=torch.autograd.grad(loss,m.parameters(),create_graph=True)
            dot=sum((a*b).sum() for a,b in zip(dg,real_grad)); dn=torch.sqrt(sum((a**2).sum() for a in dg))
            cos=1-dot/(dn*rn+1e-8); (cos+tv_weight*total_variation(dx)).backward()
            opt.step(); sched.step(); v=float(cos.detach().item())
            if np.isfinite(v) and v<l: l,i=v,torch.sigmoid(latent).detach().clone()
        if l<bL: bL,bI=l,i
    return bI

# ----------------------------- FedAvg -----------------------------
def local_train(gstate,loader,cfg):
    m=build_model(); m.load_state_dict(gstate); m.train()
    opt=torch.optim.SGD(m.parameters(),lr=cfg.lr); crit=nn.CrossEntropyLoss()
    for _ in range(cfg.local_epochs):
        for x,y in loader:
            x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad(); crit(m(x),y).backward(); opt.step()
    return m.state_dict()

def defended_local_train(gstate,loader,cfg,spec):     # per-batch defense + error feedback (pruning)
    m=build_model(); m.load_state_dict(gstate); m.train(); params=list(m.parameters()); crit=nn.CrossEntropyLoss()
    use_prune=spec.get("prune",0)>0; residual=[torch.zeros_like(p) for p in params] if use_prune else None
    for _ in range(cfg.local_epochs):
        for x,y in loader:
            x,y=x.to(DEVICE),y.to(DEVICE); loss=crit(m(x),y); grads=list(torch.autograd.grad(loss,params))
            if use_prune:
                grads=[g+r for g,r in zip(grads,residual)]; pr=prune_gradient(grads,spec["prune"])
                residual=[(g-p).detach() for g,p in zip(grads,pr)]; grads=pr
            if spec.get("clip") is not None or spec.get("sigma",0)>0:
                grads=dp_gradient(grads,spec.get("clip"),spec.get("sigma",0))
            with torch.no_grad():
                for p,g in zip(params,grads): p-=cfg.lr*g
    return m.state_dict()

def average_weights(states,sizes):
    tot=float(sum(sizes)); avg=copy.deepcopy(states[0])
    for k in avg: avg[k]=sum(s[k].float()*(n/tot) for s,n in zip(states,sizes))
    return avg

@torch.no_grad()
def evaluate(state,loader):
    m=build_model(); m.load_state_dict(state); m.eval(); c=t=0
    for x,y in loader:
        x,y=x.to(DEVICE),y.to(DEVICE); c+=(m(x).argmax(1)==y).sum().item(); t+=y.size(0)
    return c/t

print("library loaded | device =",DEVICE)


library loaded | device = cuda


In [4]:
# ==========================================================================
# STEP 1 -- FedAvg baseline + multi-stage checkpoints
# RUN THREE TIMES: WHICH="smoke" (test) -> "noniid" -> "iid".
# noniid AND iid are BOTH needed (axis 3 compares them).
# ==========================================================================
WHICH = "iid"
if   WHICH=="smoke":  cfg=Config(num_clients=4, alpha=0.5,  rounds=4,  local_epochs=1, data_fraction=0.1)
elif WHICH=="noniid": cfg=Config(num_clients=10,alpha=0.5,  rounds=60)
elif WHICH=="iid":    cfg=Config(num_clients=10,alpha=100,  rounds=60)
else: raise ValueError("WHICH must be smoke / noniid / iid")

def run_baseline(cfg):
    set_seed(cfg.seed); os.makedirs(OUT,exist_ok=True)
    tr,te=get_datasets(cfg); loaders,parts=make_client_loaders(cfg,tr)
    test_loader=DataLoader(te,batch_size=256,shuffle=False)
    print("device=",DEVICE,"alpha=",cfg.alpha,"rounds=",cfg.rounds)
    print("samples/client:",[len(p) for p in parts])
    gstate=build_model().state_dict(); sizes=[len(l.dataset) for l in loaders]
    snaps={0:copy.deepcopy(gstate)}; hist=[]; mid=max(1,cfg.rounds//2); t0=time.time()
    for r in range(1,cfg.rounds+1):
        states=[local_train(gstate,l,cfg) for l in loaders]; gstate=average_weights(states,sizes)
        acc=evaluate(gstate,test_loader); hist.append(acc)
        if r==mid: snaps[mid]=copy.deepcopy(gstate)
        if r==1 or r%5==0 or r==cfg.rounds: print("  round",r,"/",cfg.rounds,"acc=%.4f"%acc)
    snaps[cfg.rounds]=copy.deepcopy(gstate); print("  %.0fs  final acc=%.4f"%(time.time()-t0,hist[-1]))
    tag="alpha"+str(cfg.alpha)
    plt.figure(figsize=(6,4)); plt.plot(range(1,len(hist)+1),hist,marker="o",ms=3)
    plt.xlabel("round"); plt.ylabel("test acc"); plt.title("FedAvg baseline alpha="+str(cfg.alpha))
    plt.grid(alpha=.3); plt.tight_layout(); plt.savefig(OUT+"/acc_"+tag+".png",dpi=150)
    x0,y0=te[0]; x0=x0.unsqueeze(0); y0=torch.tensor([y0])
    pack={"x_true":x0,"y_true":y0,"alpha":cfg.alpha,"stages":{}}
    for r,name in {0:"untrained",mid:"mid",cfg.rounds:"final"}.items():
        st=snaps[r]; g=capture_gradient(st,x0,y0); a=evaluate(st,test_loader)
        pack["stages"][name]={"round":r,"state":st,"grad":g,"acc":a}
        print("  stage",name,"round",r,"acc=%.3f"%a)
    torch.save(pack,OUT+"/stages_"+tag+".pt"); print("saved ->",OUT+"/stages_"+tag+".pt")

run_baseline(cfg)


device= cuda alpha= 100 rounds= 60
samples/client: [5000, 5029, 5214, 4940, 4922, 5019, 4915, 4940, 5039, 4982]
  round 1 / 60 acc=0.1000
  round 5 / 60 acc=0.1000
  round 10 / 60 acc=0.1000
  round 15 / 60 acc=0.1568
  round 20 / 60 acc=0.2274
  round 25 / 60 acc=0.2845
  round 30 / 60 acc=0.2980
  round 35 / 60 acc=0.3238
  round 40 / 60 acc=0.3415
  round 45 / 60 acc=0.3629
  round 50 / 60 acc=0.3797
  round 55 / 60 acc=0.3982
  round 60 / 60 acc=0.4115
  912s  final acc=0.4115
  stage untrained round 0 acc=0.100
  stage mid round 30 acc=0.298
  stage final round 60 acc=0.411
saved -> /kaggle/working/outputs/stages_alpha100.pt


In [5]:
# ==========================================================================
# STEP 2 -- attack correctness + AXIS: TRAINING STAGE  (DLG & IG)
# Needs stages_alpha0.5.pt  (run Step 1 "noniid" first).
# ==========================================================================
pack=torch.load(OUT+"/stages_alpha0.5.pt",map_location=DEVICE)
x_true=pack["x_true"].to(DEVICE); Y=pack["y_true"]

# (1) idealized validation: untrained weights_init net -> should be near-perfect
set_seed(0); vnet=build_model(); vnet.apply(weights_init)
vgrad=capture_gradient(vnet.state_dict(),x_true,Y)
vrec=dlg_attack(vnet.state_dict(),vgrad,Y,restarts=4); vp,vs,vm=score(vrec,x_true)
print("[validation] DLG on idealized untrained: PSNR=%.1f SSIM=%.3f  (expect ~perfect)"%(vp,vs))

# (2) training-stage trajectory, BOTH attacks
rows=[]
for stage in ["untrained","mid","final"]:
    info=pack["stages"][stage]; g=info["grad"]
    dl=dlg_attack(info["state"],g,Y,restarts=4); _,ds,_=score(dl,x_true)
    ig=ig_attack(info["state"],g,Y,restarts=2); _,is_,_=score(ig,x_true)
    rows.append((stage,info["round"],info["acc"],ds,is_))
    print("  %-9s r%-2d acc=%.3f | DLG SSIM=%.3f | IG SSIM=%.3f"%(stage,info["round"],info["acc"],ds,is_))
RESULTS["training_stage"]=rows
plt.figure(figsize=(6,4))
plt.plot([r[1] for r in rows],[r[3] for r in rows],"o-",label="DLG")
plt.plot([r[1] for r in rows],[r[4] for r in rows],"s-",label="IG")
plt.xlabel("training round"); plt.ylabel("SSIM"); plt.title("Axis: training stage")
plt.grid(alpha=.3); plt.legend(); plt.tight_layout(); plt.savefig(OUT+"/axis_training_stage.png",dpi=150)
print("saved -> axis_training_stage.png")


[validation] DLG on idealized untrained: PSNR=48.3 SSIM=0.999  (expect ~perfect)
  untrained r0  acc=0.100 | DLG SSIM=0.477 | IG SSIM=0.085
  mid       r30 acc=0.381 | DLG SSIM=0.662 | IG SSIM=0.356
  final     r60 acc=0.460 | DLG SSIM=0.799 | IG SSIM=0.596
saved -> axis_training_stage.png


In [6]:
# ==========================================================================
# STEP 3 -- AXIS: BATCH SIZE  (DLG & IG, trained net; labels known for B>1)
# Needs stages_alpha0.5.pt.
# ==========================================================================
pack=torch.load(OUT+"/stages_alpha0.5.pt",map_location=DEVICE)
state=pack["stages"]["final"]["state"]; _,te=get_datasets(Config())
rows=[]
for B in [1,4,8,16]:
    X=torch.stack([te[i][0] for i in range(B)]).to(DEVICE); Yb=torch.tensor([te[i][1] for i in range(B)])
    g=capture_gradient(state,X,Yb)
    dl=dlg_attack(state,g,Yb,restarts=2); dp_,ds,_=score(dl,X)
    ig=ig_attack(state,g,Yb,iters=1200,restarts=2); ip,is_,_=score(ig,X)
    rows.append((B,dp_,ds,ip,is_))
    print("  B=%-2d | DLG SSIM=%.3f PSNR=%.1f | IG SSIM=%.3f PSNR=%.1f"%(B,ds,dp_,is_,ip))
    if B==4: RESULTS["batch4_imgs"]=(X.cpu(),dl.cpu(),ig.cpu())
RESULTS["batch_sweep"]=rows
Bs=[r[0] for r in rows]
plt.figure(figsize=(6,4)); plt.plot(Bs,[r[2] for r in rows],"o-",label="DLG"); plt.plot(Bs,[r[4] for r in rows],"s-",label="IG")
plt.xlabel("batch size"); plt.ylabel("mean SSIM"); plt.title("Axis: batch size"); plt.xticks(Bs)
plt.grid(alpha=.3); plt.legend(); plt.tight_layout(); plt.savefig(OUT+"/axis_batch_size.png",dpi=150)
print("saved -> axis_batch_size.png")


  B=1  | DLG SSIM=0.812 PSNR=21.6 | IG SSIM=0.519 PSNR=19.3
  B=4  | DLG SSIM=0.470 PSNR=15.3 | IG SSIM=0.453 PSNR=16.7
  B=8  | DLG SSIM=0.306 PSNR=13.0 | IG SSIM=0.286 PSNR=14.8
  B=16 | DLG SSIM=0.108 PSNR=7.1 | IG SSIM=0.216 PSNR=13.2
saved -> axis_batch_size.png


In [7]:
# ==========================================================================
# STEP 4 -- AXIS: IID vs NON-IID  (DLG & IG, converged net, bs=1)
# Needs BOTH stages_alpha0.5.pt and stages_alpha100.pt
# (run Step 1 with WHICH="noniid" AND WHICH="iid").
# Same victim image, same architecture; only the data partition differs.
# ==========================================================================
rows=[]
for tag,name in [("alpha0.5","non-IID (a=0.5)"),("alpha100","near-IID (a=100)")]:
    pk=torch.load(OUT+"/stages_"+tag+".pt",map_location=DEVICE)
    info=pk["stages"]["final"]; xt=pk["x_true"].to(DEVICE); Y=pk["y_true"]; g=info["grad"]
    dl=dlg_attack(info["state"],g,Y,restarts=4); _,ds,_=score(dl,xt)
    ig=ig_attack(info["state"],g,Y,restarts=2); _,is_,_=score(ig,xt)
    rows.append((name,info["acc"],ds,is_))
    print("  %-16s acc=%.3f | DLG SSIM=%.3f | IG SSIM=%.3f"%(name,info["acc"],ds,is_))
RESULTS["iid_vs_noniid"]=rows


  non-IID (a=0.5)  acc=0.460 | DLG SSIM=0.807 | IG SSIM=0.576
  near-IID (a=100) acc=0.411 | DLG SSIM=0.582 | IG SSIM=0.351


In [8]:
# ==========================================================================
# STEP 5 -- AXIS: SINGLE-STEP vs MULTI-STEP (FedSGD vs FedAvg), bs=1 victim
# Attacker reconstructs at the converged global state; victim runs E local SGD
# steps on its image and transmits the update. E=1 == single-step (exact).
# Needs stages_alpha0.5.pt.
# ==========================================================================
pack=torch.load(OUT+"/stages_alpha0.5.pt",map_location=DEVICE)
state=pack["stages"]["final"]["state"]; x0=pack["x_true"].to(DEVICE); Y=pack["y_true"]; lr=Config().lr
rows=[]
for E in [1,2,4,8]:
    pg=capture_multistep_update(state,x0,Y,local_steps=E,lr=lr)
    dl=dlg_attack(state,pg,Y,restarts=4); _,ds,_=score(dl,x0)
    ig=ig_attack(state,pg,Y,restarts=2); _,is_,_=score(ig,x0)
    rows.append((E,ds,is_)); print("  local steps E=%-2d | DLG SSIM=%.3f | IG SSIM=%.3f"%(E,ds,is_))
RESULTS["multistep"]=rows
Es=[r[0] for r in rows]
plt.figure(figsize=(6,4)); plt.plot(Es,[r[1] for r in rows],"o-",label="DLG"); plt.plot(Es,[r[2] for r in rows],"s-",label="IG")
plt.xlabel("local SGD steps (E=1 is single-step)"); plt.ylabel("SSIM"); plt.title("Axis: single vs multi-step")
plt.xticks(Es); plt.grid(alpha=.3); plt.legend(); plt.tight_layout(); plt.savefig(OUT+"/axis_multistep.png",dpi=150)
print("saved -> axis_multistep.png")


  local steps E=1  | DLG SSIM=0.799 | IG SSIM=0.569
  local steps E=2  | DLG SSIM=0.703 | IG SSIM=0.569
  local steps E=4  | DLG SSIM=0.613 | IG SSIM=0.572
  local steps E=8  | DLG SSIM=0.428 | IG SSIM=0.566
saved -> axis_multistep.png


In [9]:
# ==========================================================================
# STEP 6 -- DEFENSE x ATTACK matrix (does IG bypass clipping?)
# Needs stages_alpha0.5.pt. Populates RESULTS["ig_ssim"] for the next cell.
# ==========================================================================
pack=torch.load(OUT+"/stages_alpha0.5.pt",map_location=DEVICE)
info=pack["stages"]["final"]; state=info["state"]; base=info["grad"]; xt=pack["x_true"].to(DEVICE); Y=pack["y_true"]
defenses=[("no defense",{}),("clip C=8",{"clip":8}),("clip C=4",{"clip":4}),("clip C=2",{"clip":2}),
          ("noise s=0.01",{"sigma":0.01}),("noise s=0.1",{"sigma":0.1}),("prune 90%",{"prune":0.9}),
          ("comb p90 c8 s0.01",{"prune":0.9,"clip":8,"sigma":0.01})]
rows=[]; ig_ssim={}
for nm,spec in defenses:
    g=apply_defense(base,spec)
    dl=dlg_attack(state,g,Y,restarts=2); _,ds,_=score(dl,xt)
    ig=ig_attack(state,g,Y,restarts=2); _,is_,_=score(ig,xt)
    rows.append((nm,ds,is_)); ig_ssim[nm]=is_
    print("  %-20s DLG=%.3f  IG=%.3f"%(nm,ds,is_))
RESULTS["defense_matrix"]=rows; RESULTS["ig_ssim"]=ig_ssim
x=np.arange(len(rows)); w=0.38
plt.figure(figsize=(11,4.5)); plt.bar(x-w/2,[r[1] for r in rows],w,label="DLG"); plt.bar(x+w/2,[r[2] for r in rows],w,label="IG")
plt.xticks(x,[r[0] for r in rows],rotation=55,ha="right",fontsize=8); plt.ylabel("SSIM (lower=better defense)")
plt.title("Defense x attack"); plt.legend(); plt.tight_layout(); plt.savefig(OUT+"/defense_x_attack.png",dpi=150)
print("saved -> defense_x_attack.png")


  no defense           DLG=0.797  IG=0.567
  clip C=8             DLG=0.113  IG=0.571
  clip C=4             DLG=0.052  IG=0.554
  clip C=2             DLG=0.057  IG=0.565
  noise s=0.01         DLG=0.134  IG=0.553
  noise s=0.1          DLG=0.047  IG=0.137
  prune 90%            DLG=0.051  IG=0.183
  comb p90 c8 s0.01    DLG=0.038  IG=0.171
saved -> defense_x_attack.png


In [10]:
# ==========================================================================
# STEP 7 -- DEFENSE UTILITY (error-feedback) + privacy-utility scatter / ablation
# RUN STEP 6 FIRST (provides RESULTS["ig_ssim"]). 30 rounds, non-IID.
# ==========================================================================
cfg=Config(rounds=30); tr,te=get_datasets(cfg); test_loader=DataLoader(te,batch_size=256,shuffle=False)
def train_def(spec,cfg):
    set_seed(cfg.seed); loaders,_=make_client_loaders(cfg,tr); g=build_model().state_dict()
    sizes=[len(l.dataset) for l in loaders]; hist=[]
    for r in range(1,cfg.rounds+1):
        states=[defended_local_train(g,l,cfg,spec) for l in loaders]; g=average_weights(states,sizes)
        if r%10==0 or r==cfg.rounds: hist.append((r,evaluate(g,test_loader)))
    return hist
configs=[("no defense",{}),("clip C=4",{"clip":4}),("noise s=0.01",{"sigma":0.01}),
         ("noise s=0.1",{"sigma":0.1}),("prune 90%",{"prune":0.9}),("comb p90 c8 s0.01",{"prune":0.9,"clip":8,"sigma":0.01})]
acc={}
for nm,spec in configs:
    t0=time.time(); h=train_def(spec,cfg); acc[nm]=h[-1][1]
    print("  %-20s acc=%.4f (%.0fs)"%(nm,acc[nm],time.time()-t0))
RESULTS["defense_acc"]=acc
ig=RESULTS.get("ig_ssim",{})
if not ig:
    print("Run STEP 6 first to populate IG SSIM for the scatter.")
else:
    plt.figure(figsize=(7.5,5.5))
    for nm in acc:
        if nm in ig:
            col="tab:green" if ig[nm]<0.30 else "tab:red"; plt.scatter(acc[nm],ig[nm],s=90,c=col)
            plt.annotate(nm,(acc[nm],ig[nm]),fontsize=8,xytext=(5,5),textcoords="offset points")
    plt.gca().invert_yaxis(); plt.xlabel("test accuracy (higher=more useful) ->")
    plt.ylabel("IG SSIM (lower=more private)"); plt.title("Privacy-utility (green=stops IG)")
    plt.grid(alpha=.3); plt.tight_layout(); plt.savefig(OUT+"/privacy_utility.png",dpi=150)
    print("saved -> privacy_utility.png")
    print("")
    print("ablation (defenses that STOP IG, SSIM<0.30), best accuracy first:")
    for nm in sorted([n for n in acc if ig.get(n,1.0)<0.30], key=lambda n:-acc[n]):
        print("  %-20s acc=%.3f  IG_SSIM=%.3f"%(nm,acc[nm],ig[nm]))


  no defense           acc=0.3759 (405s)
  clip C=4             acc=0.3754 (475s)
  noise s=0.01         acc=0.3828 (426s)
  noise s=0.1          acc=0.4062 (424s)
  prune 90%            acc=0.3644 (438s)
  comb p90 c8 s0.01    acc=0.3771 (496s)
saved -> privacy_utility.png

ablation (defenses that STOP IG, SSIM<0.30), best accuracy first:
  noise s=0.1          acc=0.406  IG_SSIM=0.137
  comb p90 c8 s0.01    acc=0.377  IG_SSIM=0.171
  prune 90%            acc=0.364  IG_SSIM=0.183


In [11]:
# ==========================================================================
# STEP 8 -- CONVERGENCE ablation (60 rounds): settle "noise vs combined"
# Reuses RESULTS["ig_ssim"] from STEP 6. Retrains 4 configs at 60 rounds.
# ~55-60 min. The 30-round run had noise ahead partly via early-training
# acceleration; this checks whether that survives to convergence.
# ==========================================================================
cfg=Config(rounds=60); tr,te=get_datasets(cfg); test_loader=DataLoader(te,batch_size=256,shuffle=False)
def train_def60(spec):
    set_seed(cfg.seed); loaders,_=make_client_loaders(cfg,tr); g=build_model().state_dict()
    sizes=[len(l.dataset) for l in loaders]
    for r in range(1,cfg.rounds+1):
        states=[defended_local_train(g,l,cfg,spec) for l in loaders]; g=average_weights(states,sizes)
    return evaluate(g,test_loader)
configs=[("no defense",{}),("noise s=0.1",{"sigma":0.1}),("prune 90%",{"prune":0.9}),
         ("comb p90 c8 s0.01",{"prune":0.9,"clip":8,"sigma":0.01})]
acc60={}
for nm,spec in configs:
    t0=time.time(); acc60[nm]=train_def60(spec); print("  %-20s acc@60=%.4f (%.0fs)"%(nm,acc60[nm],time.time()-t0))
RESULTS["defense_acc60"]=acc60
ig=RESULTS.get("ig_ssim",{})
print(""); print("CONVERGED ablation (IG-stoppers, best accuracy first):")
for nm in sorted([n for n in acc60 if ig.get(n,1.0)<0.30], key=lambda n:-acc60[n]):
    print("  %-20s acc@60=%.3f  IG_SSIM=%.3f"%(nm,acc60[nm],ig.get(nm,float("nan"))))


  no defense           acc@60=0.4608 (819s)
  noise s=0.1          acc@60=0.4369 (851s)
  prune 90%            acc@60=0.4642 (871s)
  comb p90 c8 s0.01    acc@60=0.4620 (1008s)

CONVERGED ablation (IG-stoppers, best accuracy first):
  prune 90%            acc@60=0.464  IG_SSIM=0.183
  comb p90 c8 s0.01    acc@60=0.462  IG_SSIM=0.171
  noise s=0.1          acc@60=0.437  IG_SSIM=0.137


In [12]:
# ==========================================================================
# STEP 9 -- MULTI-IMAGE defense x attack matrix (mean +/- std over N victims)
# Headline rigor: makes the clip-bypass / false-security finding statistically
# solid. N_IMAGES=5 (~20 min); bump to 8-10 for more evidence (slower).
# ==========================================================================
import collections
N_IMAGES=5
pack=torch.load(OUT+"/stages_alpha0.5.pt",map_location=DEVICE)
state=pack["stages"]["final"]["state"]; _,te=get_datasets(Config())
defenses=[("no defense",{}),("clip C=4",{"clip":4}),("noise s=0.01",{"sigma":0.01}),
          ("noise s=0.1",{"sigma":0.1}),("prune 90%",{"prune":0.9}),("comb p90 c8 s0.01",{"prune":0.9,"clip":8,"sigma":0.01})]
dlg_s=collections.defaultdict(list); ig_s=collections.defaultdict(list)
for n in range(N_IMAGES):
    x=te[n][0].unsqueeze(0).to(DEVICE); y=torch.tensor([te[n][1]]); base=capture_gradient(state,x,y)
    for nm,spec in defenses:
        g=apply_defense(base,spec)
        dl=dlg_attack(state,g,y,restarts=1); _,ds,_=score(dl,x); dlg_s[nm].append(ds)
        ig=ig_attack(state,g,y,iters=1000,restarts=1); _,is_,_=score(ig,x); ig_s[nm].append(is_)
    print("  image",n,"done")
print(""); print("%-20s %16s %16s"%("defense","DLG mean+/-std","IG mean+/-std"))
rows=[]
for nm,_ in defenses:
    dm,dd=float(np.mean(dlg_s[nm])),float(np.std(dlg_s[nm])); im,idd=float(np.mean(ig_s[nm])),float(np.std(ig_s[nm]))
    rows.append((nm,dm,dd,im,idd)); print("  %-20s %6.3f+/-%.3f  %6.3f+/-%.3f"%(nm,dm,dd,im,idd))
RESULTS["defense_matrix_multi"]=rows
x=np.arange(len(rows)); w=0.38
plt.figure(figsize=(11,4.5))
plt.bar(x-w/2,[r[1] for r in rows],w,yerr=[r[2] for r in rows],capsize=3,label="DLG")
plt.bar(x+w/2,[r[3] for r in rows],w,yerr=[r[4] for r in rows],capsize=3,label="IG")
plt.xticks(x,[r[0] for r in rows],rotation=55,ha="right",fontsize=8); plt.ylabel("SSIM (lower=better defense)")
plt.title("Defense x attack  (mean over %d images)"%N_IMAGES); plt.legend(); plt.tight_layout()
plt.savefig(OUT+"/defense_x_attack_multi.png",dpi=150); print("saved -> defense_x_attack_multi.png")


  image 0 done
  image 1 done
  image 2 done
  image 3 done
  image 4 done

defense                DLG mean+/-std    IG mean+/-std
  no defense            0.784+/-0.059   0.476+/-0.053
  clip C=4              0.160+/-0.079   0.498+/-0.048
  noise s=0.01          0.146+/-0.034   0.461+/-0.039
  noise s=0.1           0.060+/-0.020   0.161+/-0.052
  prune 90%             0.096+/-0.024   0.252+/-0.049
  comb p90 c8 s0.01     0.090+/-0.031   0.243+/-0.050
saved -> defense_x_attack_multi.png


In [13]:
# ==========================================================================
# STEP 10 -- MULTI-IMAGE batch axis (mean +/- std over N independent batches)
# Stabilises the noisy DLG large-batch SSIM. N_BATCH=5 (~15 min).
# ==========================================================================
import collections
N_BATCH=5
pack=torch.load(OUT+"/stages_alpha0.5.pt",map_location=DEVICE)
state=pack["stages"]["final"]["state"]; _,te=get_datasets(Config())
res=collections.defaultdict(lambda:([],[]))
for b in range(N_BATCH):
    off=b*16
    for B in [1,4,8,16]:
        X=torch.stack([te[off+i][0] for i in range(B)]).to(DEVICE); Y=torch.tensor([te[off+i][1] for i in range(B)])
        g=capture_gradient(state,X,Y)
        dl=dlg_attack(state,g,Y,restarts=1); _,ds,_=score(dl,X)
        ig=ig_attack(state,g,Y,iters=1000,restarts=1); _,is_,_=score(ig,X)
        res[B][0].append(ds); res[B][1].append(is_)
    print("  batch-draw",b,"done")
Bs=[1,4,8,16]
print(""); print("%4s %16s %16s"%("B","DLG mean+/-std","IG mean+/-std"))
dlg_m=[float(np.mean(res[B][0])) for B in Bs]; dlg_e=[float(np.std(res[B][0])) for B in Bs]
ig_m=[float(np.mean(res[B][1])) for B in Bs]; ig_e=[float(np.std(res[B][1])) for B in Bs]
for i,B in enumerate(Bs): print("  %2d  %6.3f+/-%.3f   %6.3f+/-%.3f"%(B,dlg_m[i],dlg_e[i],ig_m[i],ig_e[i]))
RESULTS["batch_multi"]=(Bs,dlg_m,dlg_e,ig_m,ig_e)
plt.figure(figsize=(6,4))
plt.errorbar(Bs,dlg_m,yerr=dlg_e,marker="o",capsize=3,label="DLG")
plt.errorbar(Bs,ig_m,yerr=ig_e,marker="s",capsize=3,label="IG")
plt.xlabel("batch size"); plt.ylabel("mean SSIM"); plt.title("Batch axis  (mean over %d draws)"%N_BATCH)
plt.xticks(Bs); plt.grid(alpha=.3); plt.legend(); plt.tight_layout()
plt.savefig(OUT+"/axis_batch_size_multi.png",dpi=150); print("saved -> axis_batch_size_multi.png")


  batch-draw 0 done
  batch-draw 1 done
  batch-draw 2 done
  batch-draw 3 done
  batch-draw 4 done

   B   DLG mean+/-std    IG mean+/-std
   1   0.747+/-0.132    0.514+/-0.055
   4   0.441+/-0.055    0.335+/-0.042
   8   0.257+/-0.075    0.240+/-0.031
  16   0.114+/-0.027    0.183+/-0.021
saved -> axis_batch_size_multi.png


In [14]:
# ==========================================================================
# STEP 11 -- MULTI-IMAGE training-stage & multi-step axes (mean +/- std)
# N_IMAGES=5 (~25 min). Confirms the monotone training-stage trend and the
# DLG-degrades / IG-flat multi-step result hold across victims, not one image.
# ==========================================================================
import collections
N_IMAGES=5
pack=torch.load(OUT+"/stages_alpha0.5.pt",map_location=DEVICE); _,te=get_datasets(Config()); lr=Config().lr
ts_dlg=collections.defaultdict(list); ts_ig=collections.defaultdict(list)
ms_dlg=collections.defaultdict(list); ms_ig=collections.defaultdict(list)
for n in range(N_IMAGES):
    x=te[n][0].unsqueeze(0).to(DEVICE); y=torch.tensor([te[n][1]])
    for stage in ["untrained","mid","final"]:
        st=pack["stages"][stage]["state"]; g=capture_gradient(st,x,y)
        dl=dlg_attack(st,g,y,restarts=1); _,ds,_=score(dl,x); ts_dlg[stage].append(ds)
        ig=ig_attack(st,g,y,iters=1000,restarts=1); _,is_,_=score(ig,x); ts_ig[stage].append(is_)
    fst=pack["stages"]["final"]["state"]
    for E in [1,2,4,8]:
        pg=capture_multistep_update(fst,x,y,local_steps=E,lr=lr)
        dl=dlg_attack(fst,pg,y,restarts=1); _,ds,_=score(dl,x); ms_dlg[E].append(ds)
        ig=ig_attack(fst,pg,y,iters=1000,restarts=1); _,is_,_=score(ig,x); ms_ig[E].append(is_)
    print("  image",n,"done")
print(""); print("TRAINING STAGE (mean+/-std):")
for stage in ["untrained","mid","final"]:
    print("  %-9s DLG %.3f+/-%.3f | IG %.3f+/-%.3f"%(stage,np.mean(ts_dlg[stage]),np.std(ts_dlg[stage]),np.mean(ts_ig[stage]),np.std(ts_ig[stage])))
print(""); print("MULTI-STEP (mean+/-std):")
for E in [1,2,4,8]:
    print("  E=%-2d DLG %.3f+/-%.3f | IG %.3f+/-%.3f"%(E,np.mean(ms_dlg[E]),np.std(ms_dlg[E]),np.mean(ms_ig[E]),np.std(ms_ig[E])))
RESULTS["training_stage_multi"]={"dlg":{k:v for k,v in ts_dlg.items()},"ig":{k:v for k,v in ts_ig.items()}}
RESULTS["multistep_multi"]={"dlg":{k:v for k,v in ms_dlg.items()},"ig":{k:v for k,v in ms_ig.items()}}


  image 0 done
  image 1 done
  image 2 done
  image 3 done
  image 4 done

TRAINING STAGE (mean+/-std):
  untrained DLG 0.345+/-0.165 | IG 0.093+/-0.014
  mid       DLG 0.657+/-0.049 | IG 0.331+/-0.083
  final     DLG 0.768+/-0.070 | IG 0.485+/-0.047

MULTI-STEP (mean+/-std):
  E=1  DLG 0.787+/-0.049 | IG 0.488+/-0.025
  E=2  DLG 0.677+/-0.092 | IG 0.474+/-0.053
  E=4  DLG 0.565+/-0.152 | IG 0.467+/-0.061
  E=8  DLG 0.434+/-0.149 | IG 0.493+/-0.033


In [15]:
# ==========================================================================
# STEP 12 -- QUALITATIVE reconstructions (the visual proof figures)
# Regenerates reconstructed images and arranges them into grids. The defense
# grid is the key one: under clipping, DLG is noise but IG is still readable.
# Needs stages_alpha0.5.pt. ~6-8 min.
# ==========================================================================
import matplotlib.pyplot as plt
pack=torch.load(OUT+"/stages_alpha0.5.pt",map_location=DEVICE)
final=pack["stages"]["final"]["state"]; _,te=get_datasets(Config())
def img(t): return t.detach().clamp(0,1).squeeze(0).permute(1,2,0).cpu().numpy()
x=te[0][0].unsqueeze(0).to(DEVICE); y=torch.tensor([te[0][1]])

# ---- (A) DEFENSE grid: rows DLG / IG, cols = defenses (the centrepiece) ----
defs=[("original",None),("no defense",{}),("clip C=4",{"clip":4}),("noise 0.01",{"sigma":0.01}),
      ("noise 0.1",{"sigma":0.1}),("prune 90%",{"prune":0.9}),("combined",{"prune":0.9,"clip":8,"sigma":0.01})]
base=capture_gradient(final,x,y)
fig,ax=plt.subplots(2,len(defs),figsize=(1.9*len(defs),4.0))
for j,(nm,spec) in enumerate(defs):
    if spec is None:
        ax[0,j].imshow(img(x)); ax[1,j].imshow(img(x))
    else:
        g=apply_defense(base,spec)
        ax[0,j].imshow(img(dlg_attack(final,g,y,restarts=1)))
        ax[1,j].imshow(img(ig_attack(final,g,y,iters=1000,restarts=1)))
    ax[0,j].set_title(nm,fontsize=8)
    for r in (0,1): ax[r,j].set_xticks([]); ax[r,j].set_yticks([])
ax[0,0].set_ylabel("DLG",fontsize=10); ax[1,0].set_ylabel("IG",fontsize=10)
fig.suptitle("Reconstruction under each defense  (victim test[0])",fontsize=11)
plt.tight_layout(); plt.savefig(OUT+"/grid_defense.png",dpi=160); print("saved -> grid_defense.png")

# ---- (B) TRAINING-STAGE grid ----
fig,ax=plt.subplots(2,4,figsize=(8,4.0))
ax[0,0].imshow(img(x)); ax[1,0].imshow(img(x)); ax[0,0].set_title("original",fontsize=8)
for j,stage in enumerate(["untrained","mid","final"]):
    st=pack["stages"][stage]["state"]; g=capture_gradient(st,x,y)
    ax[0,j+1].imshow(img(dlg_attack(st,g,y,restarts=1)))
    ax[1,j+1].imshow(img(ig_attack(st,g,y,iters=1000,restarts=1)))
    ax[0,j+1].set_title(stage,fontsize=8)
for r in range(2):
    for j in range(4): ax[r,j].set_xticks([]); ax[r,j].set_yticks([])
ax[0,0].set_ylabel("DLG",fontsize=10); ax[1,0].set_ylabel("IG",fontsize=10)
fig.suptitle("Reconstruction across training stages",fontsize=11)
plt.tight_layout(); plt.savefig(OUT+"/grid_training_stage.png",dpi=160); print("saved -> grid_training_stage.png")

# ---- (C) MULTI-STEP grid (DLG degrades, IG flat) ----
fig,ax=plt.subplots(2,5,figsize=(10,4.0)); lr=Config().lr
ax[0,0].imshow(img(x)); ax[1,0].imshow(img(x)); ax[0,0].set_title("original",fontsize=8)
for j,E in enumerate([1,2,4,8]):
    pg=capture_multistep_update(final,x,y,local_steps=E,lr=lr)
    ax[0,j+1].imshow(img(dlg_attack(final,pg,y,restarts=1)))
    ax[1,j+1].imshow(img(ig_attack(final,pg,y,iters=1000,restarts=1)))
    ax[0,j+1].set_title("E=%d"%E,fontsize=8)
for r in range(2):
    for j in range(5): ax[r,j].set_xticks([]); ax[r,j].set_yticks([])
ax[0,0].set_ylabel("DLG",fontsize=10); ax[1,0].set_ylabel("IG",fontsize=10)
fig.suptitle("Reconstruction vs local steps (single->multi-step)",fontsize=11)
plt.tight_layout(); plt.savefig(OUT+"/grid_multistep.png",dpi=160); print("saved -> grid_multistep.png")
print("done: 3 qualitative grids saved")

saved -> grid_defense.png
saved -> grid_training_stage.png
saved -> grid_multistep.png
done: 3 qualitative grids saved
